# Case Study: Cyclistic Bike-Share

**How do annual members and casual riders use Cyclistic bikes differently?**<br>
*Google Data Analytics Professional Certificate – Course 8 Capstone*

**Shaine Meister**<br>
**March 28, 2026**

---


## Introduction
Cyclistic is a bike-share program in Chicago with more than 5,800 bicycles and 600 docking stations. The company offers traditional bikes as well as assistive options such as reclining bikes, hand tricycles, and cargo bikes. Until recently, Cyclistic’s marketing strategy focused on building general awareness with flexible pricing: single-ride passes, full-day passes, and annual memberships. Casual riders buy single-ride or full-day passes; annual members purchase yearly memberships.

Cyclistic’s finance team has shown that annual members are far more profitable than casual riders. The director of marketing, Lily Moreno, believes the company’s future success depends on converting casual riders into annual members. To design an effective marketing campaign, the team first needs clear answers to three guiding questions:

1. **How do annual members and casual riders use Cyclistic bikes differently?**
2. **Why would casual riders buy Cyclistic annual memberships?**  
3. **How can Cyclistic use digital media to influence casual riders to become members?**

This notebook follows the official Google 6-step data analysis process (**Ask**, **Prepare**, **Process**, **Analyze**, **Share**, & **Act**) exactly. All analysis is performed in Python using only **pandas** and **numpy** modules. The data is the most recent 12 months of Cyclistic trip records (public Divvy dataset, March 2025 – February 2026). No personally identifiable information is present.

**Tools & Process**  
This notebook was developed in VS Code using pandas and numpy only. AI coding assistants (Grok + VS Code Copilot) were used as a research aid to generate and refine reusable helper functions and aggregation logic; as the data analyst, every cell was manually reviewed, tested for quality, and customized for the Cyclistic dataset.

## 1. Ask
**Business task**  
Design marketing strategies aimed at converting casual riders into annual members. The first step is to understand how annual members and casual riders use Cyclistic bikes differently so that targeted campaigns can be built.

**Key stakeholders**  
- Lily Moreno – Director of Marketing  
- Cyclistic marketing analytics team  
- Cyclistic executive team  

**Guiding questions (primary focus of this analysis)**  
- How do annual members versus casual riders differ in usage?  
- Why would casual riders buy a membership?  
- How can digital media convert casual riders into members?  

This analysis directly answers the first question and provides the foundation for answering the other two.

**Success criteria**  
Identify clear, actionable differences in ride length, day-of-week patterns, rideable type usage, and station preferences between the two rider groups.

## 2. Prepare
**Data location**  
Public Cyclistic (Divvy) trip data: [divvy data link](https://divvy-tripdata.s3.amazonaws.com/index.html "https://divvy-tripdata.s3.amazonaws.com/index.html")<br>
Weather Data: [Open-Meteo's](https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv "https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv")<br>
Cleaned data source: [dataprocess](https://www.kaggle.com/code/shainemeister/case-study-1-gda-notebook-dataprocess "case-study-1-dga-notebook-dataprocess") - notebook processed from my [kaggle account](https://www.kaggle.com/code/shainemeister "https://www.kaggle.com/code/shainemeister").<br>

In this section, the notebook builds the raw trip dataset by validating the requested monthly Divvy files against the public S3 bucket, downloading ZIP archives when needed, extracting the monthly CSV files, and combining them into a single DataFrame.

**What this section does**
1. Validate the configured `start_yyyymm` to `end_yyyymm` month range against available Divvy S3 objects.
2. Download missing ZIP archives to a local cache and extract their CSV contents.
3. Concatenate all monthly CSV files into one consolidated dataset.
4. Standardize `started_at` as datetime and sort records chronologically.
5. Run a quick schema sanity check across extracted monthly files.

**How the data is organized** *(sample)* 
<div style="font-size: 10px;">

| `started_at` | `day_of_week` | `start_station_name` | `start_station_id` | `start-lat_vmap` | `start-lng_vmap` |
|---|---|---|---|---|---|
| Trip start timestamp | Num day of week (`0`=Mon, `6`=Sun) | Origin station name | Origin station identifier | Vector-mapped start latitude coordinate | Vector-mapped start longitude coordinate |

</div>

The combined trip dataset is expected to contain approximately 5-6 million rows, depending on the selected date range.

**ROCCC verification**  
- Reliable: Collected by Cyclistic's own system.  
- Original: First-party trip data.  
- Comprehensive: Covers every ride in the system.  
- Current: Uses the configured recent month range.  
- Cited: Licensed for analysis (Motivate International).  

**Licensing, privacy, and security**  
Data is public under the Divvy data license. No personally identifiable information is included, so privacy is preserved. Files are cached locally for reproducible reruns of the notebook.

**Prepare outputs produced here**  
- `df`: consolidated raw trip dataset with `started_at` parsed and sorted.  
- `schema_check`: quick cross-file schema QA summary.  

**Data integrity check**  
A quick preview of row structure and column consistency is displayed below before the notebook moves to the next section.

In [1]:
import zipfile
import numpy as np
import pandas as pd

In [3]:
with zipfile.ZipFile('/mnt/RepoRetLabs/code/jupyter-notebooks/case-study_bike-share-success/202503-202602-divvy-tripdata-enriched.zip') as z:
    csv_files = [name for name in z.namelist() if name.lower().endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV file found inside the ZIP archive.")
    df = pd.read_csv(z.open(csv_files[0]))

print(df.shape)
df.head()

(3722280, 30)


,started_at,day_of_week,start_station_name,start_station_id,start-lat_vmap,start-lng_vmap,end_station_name,end_station_id,end-lat_vmap,end-lng_vmap,...,weather_longitude,distance_miles,temperature_7ft_f,relative_humidity_7ft_pct,precipitation_in,rain_in,snowfall_in,wind_speed_33ft_mph,wind_direction_33ft_deg,cloud_cover_pct
0,2025-02-28 19:20:00,4,Sheffield Ave & Webster Ave,TA1309000033,41.921525,-87.653825,Larrabee St & Menomonee St,TA1306000007,41.914675,-87.643325,...,-87.37610,17.722198,40.46,63.0,0.0,0.0,0.0,30.074356,303.0,98.0
1,2025-02-28 22:35:00,4,Wilton Ave & Belmont Ave,TA1307000134,41.940225,-87.652925,Honore St & Division St,TA1305000034,41.903125,-87.673925,...,-87.37610,16.946416,35.42,67.0,0.0,0.0,0.0,24.295606,311.0,97.0
2,2025-02-28 23:25:00,4,Michigan Ave & Madison St,13036,41.882125,-87.625125,Wabash Ave & Adams St,KA1503000015,41.879475,-87.625675,...,-87.78903,18.003188,37.04,59.0,0.0,0.0,0.0,11.246815,307.0,100.0
3,2025-02-28 23:50:00,4,Loomis St & Lexington St,13332,41.872225,-87.661375,Green St & Washington Blvd,13053,41.883175,-87.648725,...,-87.78903,16.576208,37.04,59.0,0.0,0.0,0.0,11.246815,307.0,100.0
4,2025-02-28 23:50:00,4,Green St & Randolph St*,chargingstx3,41.883625,-87.648625,Kingsbury St & Erie St,13265,41.893925,-87.641725,...,-87.78903,17.562117,37.04,59.0,0.0,0.0,0.0,11.246815,307.0,100.0


## 3. Process  
**Cleaning Summary** (already completed in companion notebook)  
- Removed negative ride lengths and extreme outliers (MAD z-score > 3.5 by rider type)  
- Created grid keys (`start-lat_vmap`) for stable geography  
- Merged hourly weather on grid + hour  
- Added simple flags: `is_weekend`, `rainy_day`, `temp_bin`  

**Weather metrics used in analysis**  
- `temperature_7ft_f` (temperature)  
- `wind_speed_33ft_mph` (wind speed)  
- `rain_in` (rain amount)  
- `cloud_cover_pct` (cloud cover)  

**Tools Used**  
- numpy  
- pandas (all steps)  
- zipfile (for direct import from zipped CSV)  

**Documentation**  
Full technical pipeline, validation counts, and outlier logs are in the companion notebook:  
`case-study-1-gda-notebook-dataprocess.ipynb` (linked in Appendix).

---

**What the code below does**  
The cell below carries the analysis directly from the cleaned dataset produced by Process.

1. **Select analysis dataframe** — Picks `df_filtered` (outlier-removed) if available, falls back to filtering `df` on the `ride_length_outlier` flag, or uses raw `df` as a last resort.  
2. **Validate required columns** — Checks that core fields and weather fields (`member_casual`, `ride_id`, `ride_length_seconds`, `temperature_7ft_f`, `wind_speed_33ft_mph`, `rain_in`, `cloud_cover_pct`) are present before running anything.  
3. **Build missing helper fields** — Recreates `is_weekend`, `rainy_day`, `temp_bin`, and `month` if needed.  
4. **1) Usage differences** — Creates reusable table/list outputs grouped by rider type, temperature bin, and weekday/weekend with ride count and weather averages.  
5. **2) Weather sensitivity** — Creates reusable table/list outputs grouped by rider type and rainy-day flag with ride count and weather averages.  
6. **3) Seasonal patterns** — Creates reusable table/list outputs grouped by month and rider type with ride count and weather averages.  
7. **4) Top 5 start stations per rider type** — Creates reusable station summary and top-5 outputs by rider group, with weather averages for each station.  
8. **Category reuse outputs** — Builds dynamic category tables and lists for all categorical-like columns so later cells can reuse category values quickly.

In [28]:

# -------------------------------------------------------
# Dataset selection
# Prefer the outlier-filtered dataset from Process.
# Falls back to filtering df on the outlier flag column,
# or raw df if neither is available.
# -------------------------------------------------------
if 'df_filtered' in globals():
    analysis_df = df_filtered.copy()
elif 'ride_length_outlier' in df.columns:
    analysis_df = df[~df['ride_length_outlier']].copy()
else:
    analysis_df = df.copy()

# -------------------------------------------------------
# Column validation
# Ensure required analysis + weather columns exist.
# -------------------------------------------------------
required_core = [
    'member_casual', 'ride_id', 'ride_length_seconds',
    'temperature_7ft_f', 'wind_speed_33ft_mph', 'rain_in', 'cloud_cover_pct'
]
missing_core = [c for c in required_core if c not in analysis_df.columns]
if missing_core:
    raise KeyError(f"Missing required columns in analysis_df: {missing_core}")

# -------------------------------------------------------
# Helper field fallbacks
# -------------------------------------------------------
if 'is_weekend' not in analysis_df.columns:
    if 'day_of_week' not in analysis_df.columns:
        raise KeyError("Missing 'is_weekend' and 'day_of_week'")
    analysis_df['is_weekend'] = analysis_df['day_of_week'].isin([5, 6])

if 'rainy_day' not in analysis_df.columns:
    analysis_df['rainy_day'] = analysis_df['rain_in'].fillna(0) > 0

if 'temp_bin' not in analysis_df.columns:
    analysis_df['temp_bin'] = pd.cut(
        analysis_df['temperature_7ft_f'],
        bins=[-np.inf, 45, 60, 75, np.inf],
        labels=['cold', 'cool', 'mild', 'warm']
    )

if 'month' not in analysis_df.columns:
    if 'started_at' not in analysis_df.columns:
        raise KeyError("Missing 'month' and 'started_at'; cannot build seasonal summary")
    analysis_df['started_at'] = pd.to_datetime(analysis_df['started_at'], errors='coerce')
    analysis_df['month'] = analysis_df['started_at'].dt.month

print(f"Analyzing rows: {len(analysis_df):,}")

# -------------------------------------------------------
# Reusable category lists (key dimensions)
# -------------------------------------------------------
rider_type_list = sorted(analysis_df['member_casual'].dropna().astype(str).unique().tolist())
temp_bin_list = [str(x) for x in analysis_df['temp_bin'].dropna().unique().tolist()]
weekend_flag_list = sorted(analysis_df['is_weekend'].dropna().unique().tolist())
rainy_day_list = sorted(analysis_df['rainy_day'].dropna().unique().tolist())
month_list = sorted(analysis_df['month'].dropna().astype(int).unique().tolist())

# -------------------------------------------------------
# Dynamic category outputs for all categorical-like columns
# -------------------------------------------------------
categorical_value_lists = {}
categorical_value_tables = {}
for col in analysis_df.columns:
    dtype = analysis_df[col].dtype
    if dtype == 'object' or dtype.name == 'category' or dtype == 'bool':
        values = sorted([v for v in analysis_df[col].dropna().unique().tolist()])
        categorical_value_lists[col] = values
        categorical_value_tables[col] = pd.DataFrame({
            'category_value': values
        })

# -------------------------------------------------------
# Shared aggregation specs (reused across all sections)
# weather_agg: avg of all 4 weather metrics
# base_agg: ride count + avg ride duration + weather_agg
# -------------------------------------------------------
weather_agg = {
    'avg_temperature_f':  ('temperature_7ft_f',   'mean'),
    'avg_wind_speed_mph': ('wind_speed_33ft_mph',  'mean'),
    'avg_rain_inches':    ('rain_in',              'mean'),
    'avg_cloud_cover_pct':('cloud_cover_pct',      'mean'),
}
base_agg = {
    'ride_count':       ('ride_id',              'count'),
    'avg_ride_seconds': ('ride_length_seconds',  'mean'),
    **weather_agg,
}

# -------------------------------------------------------
# 1) Usage differences
# -------------------------------------------------------
usage_table = (
    analysis_df
    .groupby(['member_casual', 'temp_bin', 'is_weekend'], observed=False)
    .agg(**base_agg)
    .reset_index()
    .rename(columns={
        'member_casual': 'rider_type',
        'temp_bin': 'temperature_group',
        'is_weekend': 'weekend'
    })
    .round(2)
)
usage_list = usage_table.to_dict('records')
print("\n[1] Usage differences")
print(usage_table.head(12).to_string(index=False))

# -------------------------------------------------------
# 2) Weather sensitivity
# -------------------------------------------------------
weather_impact_table = (
    analysis_df
    .groupby(['member_casual', 'rainy_day'])
    .agg(**base_agg)
    .reset_index()
    .rename(columns={'member_casual': 'rider_type'})
    .round(2)
)
weather_impact_list = weather_impact_table.to_dict('records')
print("\n[2] Weather sensitivity")
print(weather_impact_table.to_string(index=False))

# -------------------------------------------------------
# 3) Seasonal patterns
# -------------------------------------------------------
seasonal_table = (
    analysis_df
    .groupby(['month', 'member_casual'])
    .agg(**base_agg, rainy_day_rate=('rainy_day', 'mean'))
    .reset_index()
    .rename(columns={'member_casual': 'rider_type'})
    .round(2)
)
seasonal_list = seasonal_table.to_dict('records')
print("\n[3] Seasonal patterns")
print(seasonal_table.head(12).to_string(index=False))

# -------------------------------------------------------
# 4) Station summaries + top 5 per rider type
# -------------------------------------------------------
station_col = 'start_station_name' if 'start_station_name' in analysis_df.columns else 'start_station_id'

station_stats_table = (
    analysis_df
    .groupby([station_col, 'member_casual'])
    .agg(
        ride_count=('ride_id', 'count'),
        avg_month=('month', 'mean'),
        **weather_agg,
        rainy_day_rate=('rainy_day', 'mean')
    )
    .reset_index()
    .rename(columns={
        station_col: 'start_station',
        'member_casual': 'rider_type'
    })
    .round(2)
)
station_stats_list = station_stats_table.to_dict('records')

top5_stations_table = (
    station_stats_table
    .sort_values('ride_count', ascending=False)
    .groupby('rider_type')
    .head(5)
    .sort_values(['rider_type', 'ride_count'], ascending=[True, False])
    .reset_index(drop=True)
)
top5_stations_list = top5_stations_table.to_dict('records')

station_list = sorted(station_stats_table['start_station'].dropna().astype(str).unique().tolist())

print("\n[4] Top 5 start stations by rider type")
for rider_type, rider_frame in top5_stations_table.groupby('rider_type'):
    print(f"\n  {rider_type.upper()}")
    print(rider_frame.to_string(index=False))

# -------------------------------------------------------
# Reusable variables (grouped)
# -------------------------------------------------------
# DataFrames (primary outputs)
# analysis_df: working analysis dataframe (cleaned/fallback)
# [1] usage_table: usage summary by rider_type x temperature_group x weekend
# [2] weather_impact_table: weather sensitivity summary by rider_type x rainy_day
# [3] seasonal_table: monthly seasonal summary by rider_type
# [4] station_stats_table: station-level summary by rider_type
# [4] top5_stations_table: top 5 stations per rider_type by ride_count

# DataFrames (dynamic category helpers)
# categorical_value_tables: dict of one-column DataFrames for each categorical-like column

# Lists (primary outputs)
# [1] usage_list: record-list version of usage_table
# [2] weather_impact_list: record-list version of weather_impact_table
# [3] seasonal_list: record-list version of seasonal_table
# [4] station_stats_list: record-list version of station_stats_table
# [4] top5_stations_list: record-list version of top5_stations_table

# Lists (category dimensions)
# rider_type_list: unique rider groups
# temp_bin_list: unique temperature groups
# weekend_flag_list: unique weekend flags
# rainy_day_list: unique rainy-day flags (from rain_in > 0)
# month_list: unique months present in data
# station_list: unique stations used in station outputs

# Lists (dynamic category helpers)
# categorical_value_lists: dict of unique values for each categorical-like column

# Validation and config
# required_core: required analysis/weather columns
# missing_core: missing columns list (empty when valid)
# station_col: selected station source field (name first, ID fallback)
# weather_agg: shared dict of 4 weather metric aggregations (reused in all sections)
# base_agg: shared dict of ride_count + avg_ride_seconds + weather_agg (reused in sections 1-3)
# -------------------------------------------------------


Analyzing rows: 3,463,415

[1] Usage differences
rider_type temperature_group  weekend  ride_count  avg_ride_seconds  avg_temperature_f  avg_wind_speed_mph  avg_rain_inches  avg_cloud_cover_pct
    casual              cold    False      113326            668.73              36.75               11.61              0.0                63.89
    casual              cold     True       64676            849.40              37.54               11.39              0.0                49.51
    casual              cool    False      164777            788.31              52.80               11.51              0.0                56.18
    casual              cool     True       91557            931.72              52.38               10.40              0.0                46.84
    casual              mild    False      353463            871.51              68.07                9.20              0.0                43.32
    casual              mild     True      214011            994.17              

## 4. Analyze

This section converts the processed data into clear, comparative rider-behavior findings using simple pandas group-by and aggregation.

**Step-by-step process**

1. **Prepare analysis inputs**  
   The script verifies required summary tables, makes local working copies, and standardizes rider labels (`casual`, `member`) for consistent comparisons.

2. **[1] Compare core usage behavior**  
   Calculates rider-level totals and weighted averages (ride duration, temperature, cloud cover).  
   Output focus: ride-length lift and basic weather-condition differences between rider types.

3. **[2] Measure weather sensitivity**  
   Splits rides into rainy vs non-rainy conditions and measures changes in ride volume and ride duration.  
   Output focus: how strongly each rider type reacts to rain and cloudier conditions.

4. **[3] Identify seasonal timing windows**  
   Finds each rider group’s peak month, summarizes the May–September window, and checks rainy-day-rate variation.  
   Output focus: seasonal volume and ride-length patterns by rider type.

5. **[4] Locate priority start stations**  
   Ranks top stations by rider type and calculates top-5 concentration share.  
   Output focus: geographic concentration by rider type.

6. **Package reusable outputs for Share/Act**  
   Stores results in reusable structures:  
   - `analyze_key_metrics_table`  
   - `analyze_section_summaries`  
   - `analyze_key_metrics_list`

**Why this matters**  
This workflow turns raw ride records into decision-ready comparative insights by clarifying:  
- **who** behaves differently (`casual` vs `member`),  
- **when** volume and ride length peak, and  
- **where** casual rides concentrate.

In [ ]:
# -------------------------------------------------------
# Analyze script (Sections 1-4)
# -------------------------------------------------------

# Validate required upstream tables before running section logic.
required_tables = [
    'usage_table', 'weather_impact_table', 'seasonal_table',
    'station_stats_table', 'top5_stations_table'
]
missing_tables = [t for t in required_tables if t not in globals()]
if missing_tables:
    raise KeyError(f"Missing required table(s): {missing_tables}")

# Work on local copies so downstream cells can still reuse original tables unchanged.
usage_an, weather_an, seasonal_an, station_stats_an, top5_an = (
    usage_table.copy(), weather_impact_table.copy(), seasonal_table.copy(),
    station_stats_table.copy(), top5_stations_table.copy()
)


# Standardize rider labels so member/casual filters are reliable.
def normalize_rider(series):
    """Normalize rider labels for stable comparisons (member/casual)."""
    return series.astype(str).str.strip().str.lower().str.replace(' ', '_', regex=False)


# Compute percentages safely when a denominator may be zero.
def safe_pct(numerator, denominator):
    """Return percentage while guarding against divide-by-zero."""
    return (numerator / denominator * 100.0) if denominator else 0.0


# Roll up grouped rows using ride_count as the weighting factor.
def weighted_mean(frame, value_col, weight_col='ride_count'):
    """Weighted mean helper used for rider-level rollups."""
    if frame.empty:
        return 0.0
    weights = frame[weight_col].astype(float)
    values = frame[value_col].astype(float)
    weight_sum = weights.sum()
    return float((values * weights).sum() / weight_sum) if weight_sum else 0.0


# Section display helper — consistent plain-text headings and aligned table output.
def show_section(title, df, fmt=None):
    """Print a plain-text section header and a formatted DataFrame."""
    print(f"\n{'-' * 64}")
    print(f"  {title}")
    print(f"{'-' * 64}")
    display_df = df.copy()
    if fmt:
        for col, f in fmt.items():
            if col in display_df.columns:
                display_df[col] = display_df[col].apply(lambda v: f.format(v))
    print(display_df.to_string(index=False))


# Shared config used by seasonal analysis and campaign targeting window.
month_name_map = {i: pd.Timestamp(2024, i, 1).strftime('%B') for i in range(1, 13)}
campaign_months = [5, 6, 7, 8, 9]  # May-Sep to match Analyze narrative.

# Normalize rider labels once across all tables.
for frame in [usage_an, weather_an, seasonal_an, station_stats_an, top5_an]:
    frame['rider_type'] = normalize_rider(frame['rider_type'])

# Shared rider type list — computed once and reused across all sections.
rider_types = sorted(usage_an['rider_type'].dropna().unique().tolist())

# =======================================================
# 1) Usage Differences
# Goal: quantify the overall behavior gap between casual and member riders.
# =======================================================

usage_rider_summary_rows = []
for rider in rider_types:
    rider_frame = usage_an[usage_an['rider_type'] == rider]
    usage_rider_summary_rows.append({
        'rider_type': rider,
        'total_rides': int(rider_frame['ride_count'].sum()),
        'avg_ride_seconds': round(weighted_mean(rider_frame, 'avg_ride_seconds'), 2),
        'avg_temperature_f': round(weighted_mean(rider_frame, 'avg_temperature_f'), 2),
        'avg_cloud_cover_pct': round(weighted_mean(rider_frame, 'avg_cloud_cover_pct'), 2),
    })

usage_rider_summary = pd.DataFrame(usage_rider_summary_rows)

# Keep slice variables for downstream compatibility.
casual_usage = usage_rider_summary[usage_rider_summary['rider_type'] == 'casual']
member_usage = usage_rider_summary[usage_rider_summary['rider_type'] == 'member']

# Index lookup for cleaner per-rider scalar extraction (avoids repeated .iloc[0]).
_usage_idx = usage_rider_summary.set_index('rider_type')
casual_avg_ride = float(_usage_idx.loc['casual', 'avg_ride_seconds']) if 'casual' in _usage_idx.index else 0.0
member_avg_ride = float(_usage_idx.loc['member', 'avg_ride_seconds']) if 'member' in _usage_idx.index else 0.0
duration_lift_pct = round(safe_pct(casual_avg_ride - member_avg_ride, member_avg_ride), 2)
if {'casual', 'member'}.issubset(_usage_idx.index):
    temp_delta_f = round(
        float(_usage_idx.loc['casual', 'avg_temperature_f']) - float(_usage_idx.loc['member', 'avg_temperature_f']), 2
    )
    cloud_delta_pct = round(
        float(_usage_idx.loc['casual', 'avg_cloud_cover_pct']) - float(_usage_idx.loc['member', 'avg_cloud_cover_pct']), 2
    )
else:
    temp_delta_f = cloud_delta_pct = 0.0

show_section('[1] Usage Differences by Rider Type', usage_rider_summary, fmt={
    'total_rides': '{:,.0f}',
    'avg_ride_seconds': '{:.0f} s',
    'avg_temperature_f': '{:.1f} F',
    'avg_cloud_cover_pct': '{:.1f}%',
})
print(f"\n  Ride-length lift (casual compared with member) : {duration_lift_pct:+.2f}%")
print(f"  Temperature delta (casual - member) : {temp_delta_f:+.2f} F")
print(f"  Cloud cover delta (casual - member) : {cloud_delta_pct:+.2f} percentage points")

# =======================================================
# 2) Weather Sensitivity
# Goal: compare rainy-day exposure and behavior changes by rider type.
# =======================================================

# Ensure rain flag is boolean for reliable rainy vs non-rainy slicing.
weather_an['rainy_day'] = weather_an['rainy_day'].astype(bool)
weather_sensitivity_rows = []
for rider in rider_types:
    rider_frame = weather_an[weather_an['rider_type'] == rider]
    rainy_frame = rider_frame[rider_frame['rainy_day']]
    dry_frame = rider_frame[~rider_frame['rainy_day']]

    rainy_rides = float(rainy_frame['ride_count'].sum()) if not rainy_frame.empty else 0.0
    dry_rides = float(dry_frame['ride_count'].sum()) if not dry_frame.empty else 0.0
    total_rides = rainy_rides + dry_rides

    rainy_duration = float(rainy_frame['avg_ride_seconds'].iloc[0]) if not rainy_frame.empty else 0.0
    dry_duration = float(dry_frame['avg_ride_seconds'].iloc[0]) if not dry_frame.empty else 0.0

    # Drops are measured relative to dry-day baseline to describe weather sensitivity.
    weather_sensitivity_rows.append({
        'rider_type': rider,
        'rainy_day_ride_share_pct': round(safe_pct(rainy_rides, total_rides), 2),
        'rain_vs_dry_ride_count_drop_pct': round(safe_pct(dry_rides - rainy_rides, dry_rides), 2),
        'rain_vs_dry_duration_drop_pct': round(safe_pct(dry_duration - rainy_duration, dry_duration), 2),
    })

weather_sensitivity_table = pd.DataFrame(weather_sensitivity_rows)
_weather_idx = weather_sensitivity_table.set_index('rider_type')
casual_rain_share = float(_weather_idx.loc['casual', 'rainy_day_ride_share_pct']) if 'casual' in _weather_idx.index else 0.0
member_rain_share = float(_weather_idx.loc['member', 'rainy_day_ride_share_pct']) if 'member' in _weather_idx.index else 0.0
casual_ride_drop = float(_weather_idx.loc['casual', 'rain_vs_dry_ride_count_drop_pct']) if 'casual' in _weather_idx.index else 0.0
member_ride_drop = float(_weather_idx.loc['member', 'rain_vs_dry_ride_count_drop_pct']) if 'member' in _weather_idx.index else 0.0
casual_duration_drop = float(_weather_idx.loc['casual', 'rain_vs_dry_duration_drop_pct']) if 'casual' in _weather_idx.index else 0.0
member_duration_drop = float(_weather_idx.loc['member', 'rain_vs_dry_duration_drop_pct']) if 'member' in _weather_idx.index else 0.0

show_section('[2] Weather Sensitivity by Rider Type', weather_sensitivity_table, fmt={
    'rainy_day_ride_share_pct': '{:.2f}%',
    'rain_vs_dry_ride_count_drop_pct': '{:.2f}%',
    'rain_vs_dry_duration_drop_pct': '{:.2f}%',
})
print("\n  Weather targeting notes:")
print(f"  - Casual rides skew warmer than member rides by {temp_delta_f:+.2f} F.")
print(f"  - Casual rides occur under lower cloud cover by {abs(cloud_delta_pct):.2f} percentage points.")
print(f"  - Rainy days reduce casual ride volume by {casual_ride_drop:.2f}% compared with {member_ride_drop:.2f}% for members.")
print(f"  - Rainy-day ride duration drops {casual_duration_drop:.2f}% for casual riders compared with {member_duration_drop:.2f}% for members.")
print(f"  - Rainy-day ride share remains low overall at {casual_rain_share:.2f}% for casual riders and {member_rain_share:.2f}% for members.")

# =======================================================
# 3) Seasonal Patterns
# Goal: detect peak demand timing and campaign-window behavior.
# =======================================================

seasonal_peaks_rows = []
seasonal_campaign_rows = []
rainy_day_rate_range_rows = []

for rider in rider_types:
    rider_frame = seasonal_an[seasonal_an['rider_type'] == rider]
    if rider_frame.empty:
        continue

    # Peak month by volume supports "when to target" decisions.
    peak_row = rider_frame.loc[rider_frame['ride_count'].idxmax()]
    peak_month = int(peak_row['month'])
    seasonal_peaks_rows.append({
        'rider_type': rider,
        'peak_month': peak_month,
        'peak_month_name': month_name_map.get(peak_month, str(peak_month)),
        'peak_month_ride_count': int(peak_row['ride_count']),
    })

    # Campaign window rollup supports seasonal marketing recommendations.
    campaign_frame = rider_frame[rider_frame['month'].isin(campaign_months)]
    seasonal_campaign_rows.append({
        'rider_type': rider,
        'campaign_window_rides': int(campaign_frame['ride_count'].sum()),
        'campaign_window_avg_ride_seconds': round(weighted_mean(campaign_frame, 'avg_ride_seconds'), 2),
        'campaign_window_rainy_day_rate_pct': round(float(campaign_frame['rainy_day_rate'].mean()) * 100.0, 2) if not campaign_frame.empty else 0.0,
    })

    # Range check shows how variable rain exposure is throughout the year.
    rainy_day_rate_range_rows.append({
        'rider_type': rider,
        'min_rainy_day_rate_pct': round(float(rider_frame['rainy_day_rate'].min()) * 100.0, 2),
        'max_rainy_day_rate_pct': round(float(rider_frame['rainy_day_rate'].max()) * 100.0, 2),
    })

seasonal_peaks_table = pd.DataFrame(seasonal_peaks_rows)
seasonal_campaign_table = pd.DataFrame(seasonal_campaign_rows)
rainy_day_rate_range_table = pd.DataFrame(rainy_day_rate_range_rows)

show_section('[3] Seasonal Peaks by Rider Type', seasonal_peaks_table, fmt={
    'peak_month_ride_count': '{:,.0f}',
})
show_section('[3] May through September Campaign Window', seasonal_campaign_table, fmt={
    'campaign_window_rides': '{:,.0f}',
    'campaign_window_avg_ride_seconds': '{:.0f} s',
    'campaign_window_rainy_day_rate_pct': '{:.2f}%',
})
show_section('[3] Annual Rainy-Day Rate Range', rainy_day_rate_range_table, fmt={
    'min_rainy_day_rate_pct': '{:.2f}%',
    'max_rainy_day_rate_pct': '{:.2f}%',
})

_seasonal_peaks_idx = seasonal_peaks_table.set_index('rider_type') if not seasonal_peaks_table.empty else pd.DataFrame()
casual_peak_month_name = (
    str(_seasonal_peaks_idx.loc['casual', 'peak_month_name'])
    if not isinstance(_seasonal_peaks_idx, pd.DataFrame) or 'casual' in _seasonal_peaks_idx.index
    else 'N/A'
)
member_peak_month_name = (
    str(_seasonal_peaks_idx.loc['member', 'peak_month_name'])
    if not isinstance(_seasonal_peaks_idx, pd.DataFrame) or 'member' in _seasonal_peaks_idx.index
    else 'N/A'
)

# =======================================================
# 4) Top 5 Start Stations by Rider Type
# Goal: quantify station concentration and surface top origin points.
# =======================================================

station_share_rows = []
for rider in rider_types:
    rider_total = float(station_stats_an.loc[station_stats_an['rider_type'] == rider, 'ride_count'].sum())
    rider_top5 = float(top5_an.loc[top5_an['rider_type'] == rider, 'ride_count'].sum())
    station_share_rows.append({
        'rider_type': rider,
        'top5_rides': int(rider_top5),
        'total_station_rides': int(rider_total),
        'top5_share_pct': round(safe_pct(rider_top5, rider_total), 2),
    })

station_share_table = pd.DataFrame(station_share_rows)

for rider, rider_frame in top5_an.groupby('rider_type'):
    show_section(
        f'[4] Top 5 Start Stations - {rider.title()}',
        rider_frame[['start_station', 'ride_count', 'avg_temperature_f', 'avg_cloud_cover_pct']].reset_index(drop=True),
        fmt={
            'ride_count': '{:,.0f}',
            'avg_temperature_f': '{:.1f} F',
            'avg_cloud_cover_pct': '{:.1f}%',
        }
    )

show_section('[4] Top-5 Station Concentration', station_share_table, fmt={
    'top5_rides': '{:,.0f}',
    'total_station_rides': '{:,.0f}',
    'top5_share_pct': '{:.2f}%',
})

_station_share_idx = station_share_table.set_index('rider_type') if not station_share_table.empty else pd.DataFrame()
casual_top5_share = float(_station_share_idx.loc['casual', 'top5_share_pct']) if not station_share_table.empty and 'casual' in _station_share_idx.index else 0.0
member_top5_share = float(_station_share_idx.loc['member', 'top5_share_pct']) if not station_share_table.empty and 'member' in _station_share_idx.index else 0.0

casual_top_station = 'N/A'
if not top5_an.empty and 'casual' in top5_an['rider_type'].values:
    _casual_top = top5_an[top5_an['rider_type'] == 'casual'].sort_values('ride_count', ascending=False).head(1)
    if not _casual_top.empty:
        casual_top_station = str(_casual_top['start_station'].iloc[0])

# -------------------------------------------------------
# Reusable Analyze outputs
# -------------------------------------------------------

# analyze_key_metrics_table: compact section metric/value summary for Share.
# Built from a single list accumulation — avoids repeated pd.concat overhead.
_key_metrics_rows = [
    {'section': '[1]', 'metric': 'casual_vs_member_duration_lift_pct', 'value': duration_lift_pct},
    {'section': '[1]', 'metric': 'casual_minus_member_temp_f', 'value': temp_delta_f},
    {'section': '[1]', 'metric': 'casual_minus_member_cloud_cover_pct_points', 'value': cloud_delta_pct},
]
for _, row in weather_sensitivity_table.iterrows():
    rider = row['rider_type']
    _key_metrics_rows.extend([
        {'section': '[2]', 'metric': f'{rider}_rainy_day_ride_share_pct', 'value': row['rainy_day_ride_share_pct']},
        {'section': '[2]', 'metric': f'{rider}_rain_vs_dry_ride_count_drop_pct', 'value': row['rain_vs_dry_ride_count_drop_pct']},
        {'section': '[2]', 'metric': f'{rider}_rain_vs_dry_duration_drop_pct', 'value': row['rain_vs_dry_duration_drop_pct']},
    ])

analyze_key_metrics_table = pd.DataFrame(_key_metrics_rows)

# analyze_section_summaries: section-keyed DataFrame bundle for downstream visuals.
analyze_section_summaries = {
    '[1]_usage_rider_summary': usage_rider_summary,
    '[2]_weather_sensitivity': weather_sensitivity_table,
    '[3]_seasonal_peaks': seasonal_peaks_table,
    '[3]_campaign_window': seasonal_campaign_table,
    '[3]_rainy_day_rate_range': rainy_day_rate_range_table,
    '[4]_station_share': station_share_table,
}

# analyze_key_metrics_list: record-style export of key metrics.
analyze_key_metrics_list = analyze_key_metrics_table.to_dict('records')

# Dynamic summary text variables (no hardcoded metric values).
usage_summary_line_1 = (
    f"  [1] Usage    : Casual riders average {duration_lift_pct:.1f}% longer rides than members "
    f"({casual_avg_ride:.0f} s vs {member_avg_ride:.0f} s)."
)
usage_summary_line_2 = (
    f"                 They are also more likely to ride in warmer conditions (+{temp_delta_f:.2f} F) "
    f"and under lower cloud cover ({cloud_delta_pct:+.2f} pct pts),"
)
usage_summary_line_3 = "                 showing stronger fair-weather demand."

weather_summary_line_1 = "  [2] Weather  : Casual riders show noticeably higher weather sensitivity than members."
weather_summary_line_2 = (
    f"                 On rainy days casual ride volume drops {casual_ride_drop:.2f}% "
    f"(vs {member_ride_drop:.2f}% for members)"
)
weather_summary_line_3 = (
    f"                 and ride duration drops {casual_duration_drop:.2f}% "
    f"(vs {member_duration_drop:.2f}% for members)."
)
weather_summary_line_4 = "                 Warmer, clearer conditions are the strongest windows for targeting casual riders."

seasonal_summary_line_1 = (
    f"  [3] Seasonal : Casual demand peaks in {casual_peak_month_name} while members peak in {member_peak_month_name}."
)
seasonal_summary_line_2 = "                 The May through September window captures the majority of casual volume and longest rides,"
seasonal_summary_line_3 = "                 making it the ideal period for digital campaigns and seasonal passes."

stations_summary_line_1 = (
    f"  [4] Stations : Casual rides are more concentrated in the top 5 start stations "
    f"({casual_top5_share:.2f}% of all casual rides)"
)
stations_summary_line_2 = (
    f"                 than member rides ({member_top5_share:.2f}%), with casuals clustering most at {casual_top_station}."
)

print(f"\n{'=' * 64}")
print("  Summary of Analyze Phase")
print(f"{'=' * 64}")
print(usage_summary_line_1)
print(usage_summary_line_2)
print(usage_summary_line_3)
print(weather_summary_line_1)
print(weather_summary_line_2)
print(weather_summary_line_3)
print(weather_summary_line_4)
print(seasonal_summary_line_1)
print(seasonal_summary_line_2)
print(seasonal_summary_line_3)
print(stations_summary_line_1)
print(stations_summary_line_2)
print(f"{'=' * 64}")
print("  These insights directly support the business task: casual riders are leisure-focused, weather-sensitive, and location-driven -")
print("  strong targets for membership conversion via hyper-local, fair-weather digital campaigns.")
print(f"{'=' * 64}")

# -------------------------------------------------------
# Variable list summary:
# -------------------------------------------------------
# DataFrames:
# - usage_an: local copy of usage_table used for section 1) rollups.
# - weather_an: local copy of weather_impact_table used for section 2) comparisons.
# - seasonal_an: local copy of seasonal_table used for section 3) monthly trends.
# - station_stats_an: local copy of station_stats_table used for section 4) station totals.
# - top5_an: local copy of top5_stations_table used for section 4) station detail.
# - usage_rider_summary: rider-level usage summary for section 1).
# - weather_sensitivity_table: rainy-vs-dry comparison table for section 2).
# - seasonal_peaks_table: highest-volume month by rider type for section 3).
# - seasonal_campaign_table: May-Sep campaign summary by rider type for section 3).
# - rainy_day_rate_range_table: min/max rainy-day rate by rider type for section 3).
# - station_share_table: top-5 station concentration summary for section 4).
# - analyze_key_metrics_table: compact metric/value table for downstream sharing.
#
# Helper functions:
# - normalize_rider(): standardizes rider labels for reliable filtering.
# - safe_pct(): computes percentages safely when denominators can be zero.
# - weighted_mean(): calculates ride-count-weighted averages.
# - show_section(): prints section headers and formatted plain-text tables.
#
# Config and grouping variables:
# - required_tables: upstream tables required before this cell can run.
# - missing_tables: missing required tables, if any.
# - month_name_map: converts month number to month name.
# - campaign_months: May-Sep month list used for campaign analysis.
# - rider_types: normalized rider labels reused across all sections.
#
# Row collectors and summary containers:
# - usage_rider_summary_rows: builds the section 1) summary DataFrame.
# - weather_sensitivity_rows: builds the section 2) summary DataFrame.
# - seasonal_peaks_rows: builds the peak-month summary table.
# - seasonal_campaign_rows: builds the campaign-window summary table.
# - rainy_day_rate_range_rows: builds the rainy-day range summary table.
# - station_share_rows: builds the top-5 station concentration summary.
# - _key_metrics_rows: builds analyze_key_metrics_table without repeated concat.
# - analyze_section_summaries: dictionary of section output DataFrames.
# - analyze_key_metrics_list: record-style export of key metrics.
#
# Key scalar metrics:
# - casual_avg_ride/member_avg_ride: average ride duration by rider type.
# - duration_lift_pct: percent lift in casual duration versus member duration.
# - temp_delta_f: casual-minus-member average temperature difference.
# - cloud_delta_pct: casual-minus-member average cloud-cover difference.
# - casual_rain_share/member_rain_share: share of rides happening on rainy days by rider type.
# - casual_ride_drop/member_ride_drop: rain-vs-dry ride-count decline by rider type.
# - casual_duration_drop/member_duration_drop: rain-vs-dry duration decline by rider type.
# - casual_peak_month_name/member_peak_month_name: dynamic seasonal peak month labels.
# - casual_top5_share/member_top5_share: dynamic top-5 station concentration rates.
# - casual_top_station: top casual start station by ride count.
# -------------------------------------------------------


----------------------------------------------------------------
  [1] Usage Differences by Rider Type
----------------------------------------------------------------
rider_type total_rides avg_ride_seconds avg_temperature_f avg_cloud_cover_pct
    casual   1,211,462            878 s            62.6 F               48.0%
    member   2,251,952            595 s            57.8 F               52.5%

  Ride-length lift (casual compared with member) : +47.54%
  Temperature delta (casual - member) : +4.84 F
  Cloud cover delta (casual - member) : -4.48 percentage points

----------------------------------------------------------------
  [2] Weather Sensitivity by Rider Type
----------------------------------------------------------------
rider_type rainy_day_ride_share_pct rain_vs_dry_ride_count_drop_pct rain_vs_dry_duration_drop_pct
    casual                    8.19%                          91.08%                         2.85%
    member                    8.70%                       

### Analysis Summary

The analysis of the cleaned, weather-enriched dataset (3,463,415 rows) reveals clear, tangible differences between casual and member riders across usage, weather, seasonality, and geography.

**Key Comparative Insights**

- **[1] Usage**  
  Casual riders are more likely to take **47.5 % longer rides** than members (878 s vs 595 s) and are more likely to ride in warmer conditions (+4.8 °F) and with lower cloud cover (-4.5 percentage points).

- **[2] Weather Sensitivity**  
  Casual riders show noticeably higher weather sensitivity than members. On rainy days casual ride volume drops 91.1 % and ride duration drops 2.85 % (compared with 90.5 % and 2.06 % for members). Rainy-day ride share remains low overall (~8 % to 9 %).

- **[3] Seasonal Patterns**  
  Casual demand peaks in **August** while members peak in **September**. The May through September window captures the majority of casual volume and longest rides.

- **[4] Top 5 Start Stations**  
  Casual rides are more concentrated in the top 5 start stations (**7.76 %** of all casual rides) than member rides (**4.36 %**), with casuals clustering most at **DuSable Lake Shore Dr & Monroe St**.

**Metric Equations (Reproducibility Reference)**

- **Ride Length Lift Percent**  
  `((casual_avg_ride_seconds - member_avg_ride_seconds) / member_avg_ride_seconds) * 100`

- **Rainy-Day Ride Count Drop Percent**  
  `((dry_day_ride_count - rainy_day_ride_count) / dry_day_ride_count) * 100`

- **Rainy-Day Ride Duration Drop Percent**  
  `((dry_day_avg_ride_seconds - rainy_day_avg_ride_seconds) / dry_day_avg_ride_seconds) * 100`

- **Peak Month by Rider Type**  
  `peak_month = month where ride_count is maximum`

- **Top 5 Station Share Percent**  
  `(top5_station_ride_count / total_station_ride_count) * 100`

- **Temperature Difference (Casual - Member)**  
  `casual_avg_temperature_f - member_avg_temperature_f`

- **Cloud Cover Difference (Casual - Member)**  
  `casual_avg_cloud_cover_percent - member_avg_cloud_cover_percent`



## Share

**Next: Share**  
The tables and key metrics above are now ready to be turned into clean, stakeholder-friendly visualizations using the reusable `analyze_section_summaries` and `analyze_key_metrics_table` variables.